# Part 1 - Multimodal RAG

DATA 266 Lab 2, Pair_Programming_Team_32

Answer questions from three economics PDFs using RAG with hybrid search and Gemini.

In [ ]:
# Cell 1 - Run information (required)
import sys, random, numpy as np
SEED = 266
random.seed(SEED)
np.random.seed(SEED)
print({"python": sys.version.split()[0], "seed": SEED})

In [ ]:
# Cell 2 - Install dependencies
!pip install -q pymupdf chromadb sentence-transformers rank-bm25 google-genai

In [ ]:
# Cell 3 - Setup API key and paths
import os
from kaggle_secrets import UserSecretsClient

# Get Gemini API key from Kaggle secrets
secrets = UserSecretsClient()
os.environ["GOOGLE_API_KEY"] = secrets.get_secret("GOOGLE_API_KEY")

# Set model (check available models if this fails)
os.environ["GEMINI_MODEL"] = "gemini-1.5-flash"  # Kaggle may have different models

print("API key configured")

In [ ]:
# Cell 4 - Clone repository
!git clone https://github.com/sakotiya/data266-lab2-team32.git
%cd data266-lab2-team32
!git checkout shreya  # Use the shreya branch with latest code

In [ ]:
# Cell 5 - Copy competition data to repo
import shutil
DATA_PATH = "/kaggle/input/data-266-fall-2026-lab-2-part-1-multimodal-rag"

for f in ["Document.pdf", "Document2_Supplement_A_Outturns_2011-2013.pdf", 
          "Document3_Briefing_B_Labour_Prices_Policy.pdf",
          "Questions.csv", "sample_submission.csv", "kb_manifest.json", "validate_submission.py"]:
    src = f"{DATA_PATH}/{f}"
    dst = f"part1_rag/data/{f}"
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Copied {f}")

!ls -la part1_rag/data/

In [ ]:
# Cell 6 - Ingest PDFs (extract text chunks and figures)
!python -m part1_rag.src.ingest

In [ ]:
# Cell 7 - Build search index
!python -m part1_rag.src.index

In [ ]:
# Cell 8 - Test search
import sys
sys.path.insert(0, ".")
from part1_rag.src.search import Searcher

s = Searcher()
results = s.search("What was U.S. unemployment rate in 2010?")
print(f"Found {len(results['text'])} chunks, {len(results['figures'])} figures")
for r in results['text'][:2]:
    print(f"  {r['doc_id']} p{r['page']}: {r['text'][:100]}...")

In [ ]:
# Cell 9 - Run full pipeline (generates submission.csv)
!python -m part1_rag.src.pipeline

In [ ]:
# Cell 10 - Validate submission
!python part1_rag/data/validate_submission.py part1_rag/outputs/submission.csv part1_rag/data/Questions.csv

In [ ]:
# Cell 11 - Preview submission
import pandas as pd
sub = pd.read_csv("part1_rag/outputs/submission.csv")
print(f"Submission: {len(sub)} rows")
display(sub.head(10))
print("\nStatus distribution:")
print(sub['status'].value_counts())

In [ ]:
# Cell 12 - Copy to submission location
import shutil
shutil.copy("part1_rag/outputs/submission.csv", "/kaggle/working/submission.csv")
print("Submission ready at /kaggle/working/submission.csv")